In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Implementing Priority Queues as Heaps

Ths notebook presents <em style="color:blue">heaps</em>.  We define the set $\mathcal{H}$ of heaps by induction:

  - $\texttt{Nil} \in \mathcal{H}$.
  - $\texttt{Node}(p,v,l,r) \in \mathcal{H}$ if and only if the following is true:
      * $p \leq l \;\wedge\; p \leq r$

        The priority stored at the root is less than or equal to every other priority stored in
        the heap. This condition is known as the *heap condition*.
        
        It is important to remember that we associate *high* priorities
        with *small* numbers.
      * $\mid l.\texttt{count}() - r.\texttt{count}() \mid \;\leq\, 1$

        The number of elements in the left subtree differs from the number of elements stored in
        the right subtree by at most one.
        This condition is called the  *balancing condition*.  
      * $l \in \mathcal{H} \;\wedge\; r \in \mathcal{H}$
      
        Both the left and the right subtree of a heap are themselves heaps.

The class `Heap` is a *superclass* for constructing heaps.  We will later define the classes `Nil` and `Node` that represent heaps of the form $\texttt{Nil}$ and $\texttt{Node}(p, v, l, r)$ respectively. 
* The class `Heap` has one static variable `sNodeCount` which is needed to assign unique identifiers to different nodes.  
  Every object of the class `Heap` has a uniques identifier `mID` that is stored as a member variable.  This identifier is used 
  by `graphviz` in order to display a heap as a tree.  
* In order to generate new identifiers we use the static variable `sNodeCount` as a counter.  
  Every time a new node is created this counter is incremented.  

The class `Node` represents a heap of the form $\texttt{Node}(p,v,l,r)$ where
  - $p$ is the *priority* stored as `mPriority`,
  - $v$ is the *value* stored as `mValue`,
  - $l$ is the *left subtree* stored as `mLeft`,
  - $r$ is the *right subtree* stored as `mRight`,
  - The total number of nodes of the tree is stored in the member variable `mCount`.

In [ ]:
class Heap<T> {
  static sNodeCount = 0;
  mID: string;

  constructor() {
    Heap.sNodeCount += 1;
    this.mID = String(Heap.sNodeCount);
  }

  getID(): string {
    return this.mID;
  }

  _count(): number {
    return 0;
  }
}

interface Heap<T> {
  _makeString(attributes: string[]): string;
  toDot(): Promise<void>;
  top(): [number, T] | undefined;
  insert(p: number, v: T): Node<T>;
  remove(): Heap<T>;
}
class Node<T> extends Heap<T> {
  mPriority: number;
  mValue: T;
  mLeft: Heap<T>;
  mRight: Heap<T>;
  mCount: number;

  constructor(priority: number, value: T, left: Heap<T>, right: Heap<T>) {
    super();
    this.mPriority = priority;
    this.mValue = value;
    this.mLeft = left;
    this.mRight = right;
    this.mCount = left._count() + right._count() + 1;
  }

  _extract(): [number, T, Heap<T>, Heap<T>] {
    return [this.mPriority, this.mValue, this.mLeft, this.mRight];
  }

  _count(): number {
    return this.mCount;
  }

  toString(): string {
    return this._makeString(["mPriority", "mValue", "mLeft", "mRight"]);
  }
}
interface Node<T> {
    top(): [number, T];
    insert(p: number, v: T): Node<T>;
    remove(): Heap<T>;
}

The function `_makeString` is a small utility used to simplify `toString()` implementations.
  - `this` is the object that is to be rendered as a string
  - `attributes` is an array of the names of properties whose values will be displayed.

It returns a string that contains the class name and the listed fields, joined with commas.

In [ ]:
Heap.prototype._makeString = function<T>(this: Heap<T>, attributes: string[]): string {
  const name = this.constructor.name;
  const content = attributes
    .map((at) => `${at}=${(this)[at]}`)
    .join(", ");
  return `${name}(${content})`;
};

## Defining $\texttt{Nil}$ and $\texttt{Node}(p, v, l, r)$ as Classes

The class `Nil` represents an empty heap.  
It has no data members and provides:
* `_count` returns the number of nodes in the tree, which is `0`.
* `toString()` returns the string `"Nil()"`.

In [ ]:
class Nil<T> extends Heap<T> {
  _count(): number {
    return 0;
  }

  toString(): string {
    return "Nil()";
  }
}
interface Nil<T> {
  top(): undefined;
  insert(p: number, v: T): Node<T>;
  remove(): Heap<T>;
}

## Graphical Representation

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

The method `_collectIDs(dict)` takes a binary trie $t$ and a dictionary $d$ and updates the dictionary $d$ so that the following holds:
$$ d[\texttt{id}] = n \quad \text{for every node } n \text{ in } t \text{ such that } \texttt{id} \text{ is the identifier of node } n.$$
It assigns a unique identifier to every node so edges can be drawn correctly.


In [ ]:
function _collectIDs<T>(self: Heap<T>, nodeDict: Record<string, Heap<T>>): void {
  nodeDict[self.getID()] = self;
  if (self instanceof Node) {
    _collectIDs(self.mLeft, nodeDict);
    _collectIDs(self.mRight, nodeDict);
  }
}

The method $t.\texttt{toDot}()$ converts a heap into a directed graph in DOT format. The DOT graph can be rendered visually via **Graphviz**  

In [ ]:
async function toDot<T>(t: Heap<T>): Promise<string> {
  const parts: string[] = [];
  parts.push('digraph G { node [shape="record", style="rounded"];');

  const nodeDict: Record<string, Heap<T>> = {};
  _collectIDs(t, nodeDict);

  for (const [id, node] of Object.entries(nodeDict)) {
    if (node instanceof Nil) {
      parts.push(`${id} [label="", shape="point"];`);
    } else if (node instanceof Node) {
      const label =
        node.mValue !== undefined
          ? `{ ${node.mPriority} | ${node.mValue} }`
          : `${node.mPriority}`;
      parts.push(`${id} [label="${label}"];`);
    }
  }

  for (const [id, node] of Object.entries(nodeDict)) {
    if (node instanceof Node) {
      parts.push(`${id} -> ${node.mLeft.getID()};`);
      parts.push(`${id} -> ${node.mRight.getID()};`);
    }
  }

  parts.push("}");
  return parts.join("\n");
}

Heap.prototype.toDot = async function<T>(this: Heap<T>): Promise<void> {
  const dot = await toDot(this);
  const gv = await Graphviz.load();
  const svg = gv.layout(dot, "svg", "dot");
  display.html(svg);
  display.html(`
      <style>
        div.output_scroll {
          height: auto !important;
        }
      </style>
  `);
};

## Implementing the Method  `top`

For the class `Nil`, the function `top`is specified via a single equation:
 - $\texttt{Nil}.\texttt{top}() = \Omega$

In [ ]:
Nil.prototype.top = function(): undefined {
  return undefined;
};

For the class `Node`, the function `top` is specified via the following equation:
- $\texttt{Node}(p,v,l,r).\texttt{top}() = \langle p,v \rangle$

In [ ]:
Node.prototype.top = function<T>(): [number, T] {
  return [this.mPriority, this.mValue];
};

## Implementing the method `insert`

The method `insert(p, v)` takes two parameters:
* `p` is a priority.  
* `v` is some value that has the priority `p`. 

The method returns the heap that results from inserting the value `v` with priority `p` into the heap.

In the class `Nil` the method `insert` is specified as follows:
- $\texttt{Nil}.\texttt{insert}(p,v) = \texttt{Node}(p,v,\texttt{Nil}, \texttt{Nil})$


In [ ]:
Nil.prototype.insert = function<T>(p: number, v: T): Node<T> {
  return new Node(p, v, new Nil(), new Nil());
};

In the class `Node` the method `insert` is specified as follows:
  - $p_{\mathrm{top}} \leq p \;\wedge\; l.\texttt{count}() \leq r.\texttt{count}() \;\rightarrow\; 
     \texttt{Node}(p_{\mathrm{top}},v_\mathrm{top},l,r).\texttt{insert}(p,v) =
     \texttt{Node}\bigl(p_\mathrm{top},v_\mathrm{top},l.\texttt{insert}(p,v), r\bigr)$               
  - $p_{\mathrm{top}} \leq p \;\wedge\; l.\texttt{count}() > r.\texttt{count}() \;\rightarrow 
     \texttt{Node}(p_{\mathrm{top}},v_\mathrm{top},l,r).\texttt{insert}(p,v) =
     \texttt{Node}\bigl(p_\mathrm{top},v_\mathrm{top},l,r.\texttt{insert}(p,v)\bigr)$
  - $p_{\mathrm{top}} > p \;\wedge\; l.\texttt{count}() \leq r.\texttt{count}() \;\rightarrow 
     \texttt{Node}(p_{\mathrm{top}},v_\mathrm{top},l,r).\texttt{insert}(p,v) =
     \texttt{Node}\bigl(p,v,l.\texttt{insert}(p_\mathrm{top},v_\mathrm{top}), r\bigr)$
  - $p_{\mathrm{top}} > p \;\wedge\; l.\texttt{count}() > r.\texttt{count}() \;\rightarrow 
     \texttt{Node}(p_{\mathrm{top}},v_\mathrm{top},l,r).\texttt{insert}(p,v) =
     \texttt{Node}\bigl(p,v,l,r.\texttt{insert}(p_\mathrm{top},v_\mathrm{top})\bigr)$

In [ ]:
Node.prototype.insert = function<T>(p: number, v: T): Node<T> {
  const [p_top, v_top, l, r] = this._extract();
  if (p_top <= p) {
    if (l._count() <= r._count()) return new Node(p_top, v_top, l.insert(p, v), r);
    else return new Node(p_top, v_top, l, r.insert(p, v));
  } else {
    if (l._count() <= r._count()) return new Node(p, v, l.insert(p_top, v_top), r);
    else return new Node(p, v, l, r.insert(p_top, v_top));
  }
};

## Implementing the Method  `remove`

The method `remove()` returns the heap that results if the value with the highest priority in the heap is removed.

In the class `Nil` the method `remove` is specified as follows:
- $\texttt{Nil}.\texttt{remove}() = \texttt{Nil}$


In [ ]:
Nil.prototype.remove = function<T>(): Nil<T> {
  return this;
};

In the class `Node` the method `remove` is specified as follows:
- $\texttt{Node}(p,v,\texttt{Nil},r).\texttt{remove}() = r$
- $\texttt{Node}(p,v,l,\texttt{Nil}).\texttt{remove}() = l$
- $l = \texttt{Node}(p_1,v_1,l_1,r_1) \;\wedge\; r = \texttt{Node}(p_2,v_2,l_2,r_2) \;\wedge\; p_1 \leq p_2 \;\rightarrow
       \texttt{Node}(p,v,l,r).\texttt{remove}() =      \texttt{Node}(p_1,v_1,l.\texttt{remove}(),r)$
- $l = \texttt{Node}(p_1,v_1,l_1,r_1) \;\wedge\; r = \texttt{Node}(p_2,v_2,l_2,r_2) \;\wedge\; p_1 > p_2 \rightarrow
       \texttt{Node}(p,v,l,r).\texttt{remove}() = \texttt{Node}(p_2,v_2,l,r.\texttt{remove}())$
       
Note that the heap returned by the method `remove` might violate the balancing condition.  As heaps only shrink when `remove` is called,
this fact does not cause performance problems.

In [ ]:
Node.prototype.remove = function<T>(): Heap<T> {
  const [p, v, l, r] = this._extract();
  if (l instanceof Nil) return r;
  if (r instanceof Nil) return l;

  const [p1, v1] = l.top()!;
  const [p2, v2] = r.top()!;

  if (p1 <= p2) return new Node(p1, v1, l.remove(), r);
  else return new Node(p2, v2, l, r.remove());
};

## Testing

Below are some simple tests to check whether the methods `insert` and `delete` work as expected. 

In [ ]:
let h: Heap<string> = new Nil<string>();
await h.toDot();

In [ ]:
h = h.insert(2, "a");
await h.toDot();

In [ ]:
h = h.insert(1, "b");
await h.toDot();

In [ ]:
h = h.insert(7, "c");
await h.toDot();

In [ ]:
h = h.insert(0, "d");
await h.toDot();

In [ ]:
h = h.insert(8, "e");
await h.toDot();

In [ ]:
h = h.insert(3, "f");
await h.toDot();

In [ ]:
h = h.insert(4, "g");
await h.toDot();

In [ ]:
h = h.remove();
await h.toDot();

Note that the balancing condition is violated above.

In [ ]:
h = h.remove();
await h.toDot();

In [ ]:
h = h.remove();
await h.toDot();

In [ ]:
h = h.remove();
await h.toDot();

In [ ]:
h = h.remove();
await h.toDot();

In [ ]:
h = h.remove();
await h.toDot();

In [ ]:
h = new Nil();
for (let i = 1; i <= 41; i++) {
  h = h.insert(i, undefined);
}
await h.toDot();

In [ ]:
function randomIntRange(min: number, max: number): number {
  return Math.floor(Math.random() * (max - min)) + min;
} 

In [ ]:
h = new Nil();
for (let i = 0; i < 32; i++) {
  const k = randomIntRange(1, 100);
  h = h.insert(k, undefined);
}
await h.toDot();

## Heapsort

Given an array `L`, the function `heapSort(L, visualize)` returns a sorted version of `L`.  The algorithm works in two phases.
- In the first phase, all elements of the array `L` are inserted as priorities into the heap `H`, which is initially empty.
  If the parameter `visualize` is `True`, then the heap `H`is displayed as a tree.

- In the second phase, the elements of `H` are extracted one by one beginning with the smallest elements.  These elements
  are appended to the array `S`, which is initially empty. When the function returns, `S` contains all the elements of `L`
  and these elements are ascending in `S`.

In [ ]:
function heapSort(L: number[], visualize = false): number[] {
  let H: Heap<number> = new Nil<number>();
  for (const p of L) {
    H = H.insert(p, undefined);
  }
  const S: number[] = [];
  if (visualize) H.toDot();
  while (H instanceof Node) {
    const [p] = H.top();
    S.push(p);
    H = H.remove();
  }
  return S;
}

In [ ]:
heapSort([77, 54, 68, 7, 13, 1, 4, 5, 6, 3, 12, 67, 12, 14, 23, 54, 67], true);

## Some More Testing

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

The function `isOrdered(L)` checks that the array `L` is sorted descendingly.

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `sameElements(L, S)` returns `true` if the array `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function $\texttt{testSort}(n, k)$ generates $n$ random arrays of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => Math.floor(Math.random() * (2 * k)));
    const oldL = L.slice();
    const S = heapSort(L);
    isOrdered(S);
    sameElements(S, oldL);
    process.stdout.write(".");
  }
  console.log();
  console.log("All tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 5_000);
console.timeEnd("testSort");

Next, we sort hundred thousand random integers.  As we have stored information for displaying heaps a s trees, the function `heapSort` isn't really competitive.

In [ ]:
console.time("heapSort 1,000,000");
const k = 1_000_000;
const L = Array.from({ length: k }, () => Math.floor(Math.random() * (2 * k)));
heapSort(L);
console.timeEnd("heapSort 1,000,000");